In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/phishing_site_urls.csv")

print("Kształt (wiersze, kolumny):", df.shape)
print("Nazwy kolumn:", df.columns.tolist())
df.head()

Kształt (wiersze, kolumny): (549346, 2)
Nazwy kolumn: ['URL', 'Label']


,URL,Label
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,bad
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,bad
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,bad
3,mail.printakid.com/www.online.americanexpress....,bad
4,thewhiskeydregs.com/wp-content/themes/widescre...,bad


In [2]:
print(df["Label"].value_counts())
print()
print((df["Label"].value_counts(normalize=True) * 100).round(2))

Label
good    392924
bad     156422
Name: count, dtype: int64

Label
good    71.53
bad     28.47
Name: proportion, dtype: float64


In [3]:
print(df.isnull().sum())

URL      0
Label    0
dtype: int64


In [4]:
print("Duplikaty (cały wiersz):", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)
print("Nowy kształt:", df.shape)

Duplikaty (cały wiersz): 42150
Nowy kształt: (507196, 2)


In [5]:
df["Label"] = df["Label"].map({"good": 0, "bad": 1})

print("Brakujące po mapowaniu:", df["Label"].isnull().sum())
print(df["Label"].value_counts())

Brakujące po mapowaniu: 0
Label
0    392897
1    114299
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

# Tasowanie, bo plik jest posortowany według etykiety
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

X = df["URL"]
y = df["Label"]

# Najpierw odcinamy 15% na test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

# Z pozostałych 85% odcinamy validation (cel: 15% całości)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=42
)

print("Train:", len(X_train), "| Val:", len(X_val), "| Test:", len(X_test))
print("Udział phishingu w train/val/test:",
      round(y_train.mean(), 3), round(y_val.mean(), 3), round(y_test.mean(), 3))

Train: 355024 | Val: 76092 | Test: 76080
Udział phishingu w train/val/test: 0.225 0.225 0.225


In [7]:
def domena(url):
    return url.split("/")[0].lower()

domeny_train = set(X_train.apply(domena))
domeny_test = set(X_test.apply(domena))
wspolne = domeny_train & domeny_test

print("Domeny w test:", len(domeny_test))
print("Wspólne domeny train i test:", len(wspolne))
print("Udział wspólnych domen w test:", round(len(wspolne) / len(domeny_test) * 100, 1), "%")

Domeny w test: 41081
Wspólne domeny train i test: 17499
Udział wspólnych domen w test: 42.6 %


In [8]:
from sklearn.model_selection import GroupShuffleSplit

def domena(url):
    return url.split("/")[0].lower()

df["Domena"] = df["URL"].apply(domena)

# Najpierw 15% domen na test
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
temp_idx, test_idx = next(gss.split(df, groups=df["Domena"]))
df_temp, df_test = df.iloc[temp_idx], df.iloc[test_idx]

# Z pozostałych 85% wydzielamy validation (cel: 15% całości)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)
train_idx, val_idx = next(gss2.split(df_temp, groups=df_temp["Domena"]))
df_train, df_val = df_temp.iloc[train_idx], df_temp.iloc[val_idx]

X_train, y_train = df_train["URL"], df_train["Label"]
X_val, y_val = df_val["URL"], df_val["Label"]
X_test, y_test = df_test["URL"], df_test["Label"]

print("Train:", len(X_train), "| Val:", len(X_val), "| Test:", len(X_test))
print("Udział phishingu:", round(y_train.mean(), 3), round(y_val.mean(), 3), round(y_test.mean(), 3))

Train: 366148 | Val: 70062 | Test: 70986
Udział phishingu: 0.216 0.244 0.254


In [9]:
domeny_train = set(X_train.apply(domena))
domeny_test = set(X_test.apply(domena))
wspolne = domeny_train & domeny_test

print("Domeny w test:", len(domeny_test))
print("Wspólne domeny train i test:", len(wspolne))

Domeny w test: 28544
Wspólne domeny train i test: 0


In [10]:
pd.DataFrame({"URL": X_train, "Label": y_train}).to_csv("../data/processed/train.csv", index=False)
pd.DataFrame({"URL": X_val, "Label": y_val}).to_csv("../data/processed/val.csv", index=False)
pd.DataFrame({"URL": X_test, "Label": y_test}).to_csv("../data/processed/test.csv", index=False)

print("Zapisano: train.csv, val.csv, test.csv")

Zapisano: train.csv, val.csv, test.csv
